In [ ]:
#!/usr/bin/env python
# coding: utf-8

# =========================================================
# IMPORT LIBRARIES
# =========================================================

import os
import sys
import json
import pandas as pd
import numpy as np

from sqlalchemy import text
from sqlalchemy.types import Text

# Add library and Database paths
sys.path.append('/home/jupyter-shared/.local_libs')
sys.path.append('/home/jupyter-shared/secrets')

from db_sql import db_analytics, db_qapp_production


# =========================================================
# CREATE DATABASE CONNECTIONS
# =========================================================

engine_destination = db_analytics()


# =========================================================
# FETCH DATA FROM MYSQL
# =========================================================

main_wcc_query = """
SELECT * 
FROM quest_analytics.main_wcc
"""

df = pd.read_sql(main_wcc_query, con=engine_destination)


# =========================================================
# BUILD JSON-READY DATAFRAME
# =========================================================

user_cols = [
    "tlo_users_id",
    "gender",
    "created_at",
    "centre_name",
    "org_name",
    "state_name",
    "district_name",
    "trade",
    "batch_name",
    "batch_status",
    "centre_type",
    "user_type",
    "platform",
    "is_ple",
    "ple_enabled",
    "a_overa_less_asses_c",
    "a_overa_assess_c",
    "a_overa_lesson_c",
    "c_overa_less_asses_c",
    "c_overa_asse_c",
    "c_overa_less_c",
    "first_login",
    "rounded_completion"
]

project_phase_cols = [
    "prog_name",
    "project_id",
    "proj_name",
    "p_phase_id",
    "phase"
]

subject_cols = [
    "sub_id",
    "sub_name",
    "avg_score_a",
    "avg_rating_a",
    "c_sub_w_less_asse_c",
    "a_sub_w_less_asse_c",
    "a_sub_w_assess_c",
    "a_sub_w_lesson_c",
    "c_sub_w_assess_c",
    "c_sub_w_less_c",
    "year_category"
]


# ---------------------------------------------------------
# Helper function to safely convert pandas/numpy values
# into JSON-compatible Python values
# ---------------------------------------------------------

def make_json_safe(value):
    if pd.isna(value):
        return None

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    return value


def records_to_json(records_df):
    records = []

    for row in records_df.to_dict("records"):
        clean_row = {
            key: make_json_safe(value)
            for key, value in row.items()
        }
        records.append(clean_row)

    return json.dumps(records, ensure_ascii=False)


# ---------------------------------------------------------
# 1. One row per user for non-repeating user-level columns
# ---------------------------------------------------------

user_df = df[user_cols].drop_duplicates(subset=["tlo_users_id"]).copy()


# ---------------------------------------------------------
# 2. Create project-phase combinations per user as JSON text
# ---------------------------------------------------------

project_phase_df = (
    df[["tlo_users_id"] + project_phase_cols]
    .drop_duplicates()
    .groupby("tlo_users_id")[project_phase_cols]
    .apply(records_to_json)
    .reset_index(name="project_phase_combos")
)


# ---------------------------------------------------------
# 3. Create subject combinations per user as JSON text
# ---------------------------------------------------------

subject_df = (
    df[["tlo_users_id"] + subject_cols]
    .drop_duplicates()
    .groupby("tlo_users_id")[subject_cols]
    .apply(records_to_json)
    .reset_index(name="subject_combos")
)


# ---------------------------------------------------------
# 4. Merge everything into final dataframe
# ---------------------------------------------------------

final_df = (
    user_df
    .merge(project_phase_df, on="tlo_users_id", how="left")
    .merge(subject_df, on="tlo_users_id", how="left")
)

# Replace NaN with None so SQL inserts NULL instead of nan
final_df = final_df.replace({np.nan: None})

print(final_df.head())

# Replace empty strings in JSON columns with None so they store as SQL NULL
for col in ["project_phase_combos", "subject_combos"]:
    final_df[col] = final_df[col].replace("", None)

# =========================================================
# WRITE BACK TO DATABASE
# =========================================================

TARGET_TABLE = "main_wcc_json"
TARGET_SCHEMA = "quest_analytics"

print(f"\nWriting to {TARGET_SCHEMA}.{TARGET_TABLE} ...")

chunksize = 10_000
total_written = 0

dtype_mapping = {
    "project_phase_combos": Text(),
    "subject_combos": Text()
}

for start in range(0, len(final_df), chunksize):
    chunk = final_df.iloc[start:start + chunksize].copy()

    chunk.to_sql(
        name=TARGET_TABLE,
        con=engine_destination,
        schema=TARGET_SCHEMA,
        if_exists="replace" if start == 0 else "append",
        index=False,
        method="multi",
        dtype=dtype_mapping
    )

    total_written += len(chunk)
    print(f"  Written {total_written:,} rows...")

print(f"\nDone. {total_written:,} rows written to {TARGET_SCHEMA}.{TARGET_TABLE}")